# Profiling de Datos - Informe de Calidad

**Autor:** Data Engineer TransCore  
**Fecha:** 2026-05-03  
**Módulo:** 06 - Ingestión y Procesamiento de Datos

---

## Objetivo

Este notebook implementa un pipeline completo de **profiling de datos** que:

1. **Lee datos desde S3 landing zone** (resultado de la ingesta previa)
2. **Genera informes de calidad** con:
   - Metadata del dataset
   - Resumen ejecutivo con score de calidad
   - Análisis detallado por columna (estadísticas descriptivas)
   - Identificación de hallazgos de calidad (nulos, valores fuera de rango, inconsistencias)
3. **Exporta resultados** a JSON y Markdown
4. **Guarda el informe en S3** junto a los datos analizados

> ⚠️ **Prerequisito:** Ejecutar primero el notebook `01.ingesta.ipynb` para tener datos en S3 landing

---

## 1. Importar Librerías

Importamos las librerías necesarias para:
- Lectura de datos desde S3 (boto3, awswrangler)
- Análisis de datos (pandas, numpy)
- Generación de informes (json, logging)

### 1.1 Configurar Credenciales AWS

Configuramos las credenciales de AWS para acceder a S3.

> ⚠️ **Importante:** Reemplaza las credenciales con las tuyas propias. Estas son las credenciales del curso proporcionadas por el formador.

In [ ]:
# Credenciales de AWS (reemplaza con tus propias credenciales)
import os
os.environ["AWS_ACCESS_KEY_ID"] = "<TU_ACCESS_KEY_ID>"
os.environ["AWS_SECRET_ACCESS_KEY"] = "<TU_SECRET_ACCESS_KEY>"

# Limpiar cualquier configuración de LocalStack
os.environ.pop("AWS_ENDPOINT_URL", None)
os.environ.pop("AWS_ENDPOINT", None)

# Configurar región de AWS explícitamente
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"
print("✅ Configuración AWS establecida: AWS real, región eu-west-1")

### 1.2 Instalar Dependencias

Primero instalamos las librerías necesarias que no vienen por defecto.

In [ ]:
# Instalar dependencias necesarias
# Ejecutar esta celda solo la primera vez
! pip install awswrangler boto3 pandas numpy -q

### 1.2 Importar Librerías

Una vez instaladas las dependencias, importamos las librerías.

In [ ]:
import json
import logging
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Any

import boto3
import numpy as np
import pandas as pd
import awswrangler as wr

# Configuración de logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

## 2. Clase DataProfiler

La clase `DataProfiler` encapsula toda la lógica de profiling:

- **Metadata**: Información básica del dataset
- **Análisis por columna**: Estadísticas según tipo de dato
- **Resumen ejecutivo**: Score de calidad global
- **Hallazgos**: Identificación automática de problemas de calidad
- **Exportación**: JSON y Markdown

In [ ]:
class DataProfiler:
    """Genera informes de profiling y calidad de datos."""
    
    def __init__(self, df: pd.DataFrame, source_filename: str):
        self.df = df
        self.source_filename = source_filename
        self.report_date = datetime.utcnow().isoformat()
        
    def generar_metadata(self) -> Dict[str, Any]:
        """Genera metadata del informe."""
        return {
            "nombre_archivo": self.source_filename,
            "fecha_generacion": self.report_date,
            "num_registros": len(self.df),
            "num_columnas": len(self.df.columns),
            "columnas": list(self.df.columns)
        }

### 2.1 Análisis por Columna

Método que analiza cada columna individualmente, detectando tipo de dato, nulos, valores únicos y estadísticas descriptivas.

In [ ]:
def analizar_columna(self, col_name: str) -> Dict[str, Any]:
    """Analiza una columna individual."""
    col_data = self.df[col_name]
    dtype = str(col_data.dtype)
    
    # Valores nulos
    nulos = col_data.isna().sum()
    pct_nulos = (nulos / len(col_data)) * 100 if len(col_data) > 0 else 0
    
    # Únicos
    num_unicos = col_data.nunique()
    
    # Completitud
    completitud = 100 - pct_nulos
    
    analisis = {
        "nombre": col_name,
        "tipo_dato": dtype,
        "nulos_count": int(nulos),
        "nulos_pct": round(pct_nulos, 2),
        "unicos": int(num_unicos),
        "completitud_pct": round(completitud, 2)
    }
    
    # Análisis según tipo
    if pd.api.types.is_numeric_dtype(col_data):
        analisis.update({
            "min": float(col_data.min()) if not pd.isna(col_data.min()) else None,
            "max": float(col_data.max()) if not pd.isna(col_data.max()) else None,
            "media": float(col_data.mean()) if not pd.isna(col_data.mean()) else None,
            "mediana": float(col_data.median()) if not pd.isna(col_data.median()) else None,
            "std_dev": float(col_data.std()) if not pd.isna(col_data.std()) else None
        })
    else:
        # Análisis categórico
        top_values = col_data.value_counts().head(5)
        analisis["top_5_valores"] = [
            {"valor": str(k), "conteo": int(v)} 
            for k, v in top_values.items()
        ]
    
    return analisis

# Añadir método a la clase
DataProfiler.analizar_columna = analizar_columna

### 2.2 Resumen Ejecutivo

Calcula métricas agregadas del dataset completo, incluyendo el **score de calidad** (0-100).

In [ ]:
def calcular_resumen_ejecutivo(self) -> Dict[str, Any]:
    """Calcula el resumen ejecutivo del dataset."""
    total_registros = len(self.df)
    total_columnas = len(self.df.columns)
    
    # Completitud global
    matriz_nulos = self.df.isna()
    total_celdas = total_registros * total_columnas
    celdas_nulas = matriz_nulos.sum().sum()
    completitud_global = ((total_celdas - celdas_nulas) / total_celdas) * 100
    
    # Score de calidad (0-100)
    score_calidad = round(completitud_global, 1)
    
    return {
        "total_registros": total_registros,
        "total_columnas": total_columnas,
        "completitud_global_pct": round(completitud_global, 2),
        "score_calidad": score_calidad
    }

# Añadir método a la clase
DataProfiler.calcular_resumen_ejecutivo = calcular_resumen_ejecutivo

### 2.3 Identificación de Hallazgos

Detecta automáticamente problemas de calidad específicos del dominio:

- ✅ Registros sin `id_equipo` (campo requerido)
- ✅ Inconsistencias entre `fecha_fin` y `estado`
- ✅ `horas_trabajo` fuera del rango válido [0-24]
- ✅ `tipo_mantenimiento` con valores no válidos

In [ ]:
def identificar_hallazgos(self) -> List[Dict[str, Any]]:
    """Identifica hallazgos de calidad específicos."""
    hallazgos = []
    
    # Hallazgo 1: Registros sin id_equipo
    sin_equipo = self.df["id_equipo"].isna().sum()
    if sin_equipo > 0:
        hallazgos.append({
            "tipo": "VALOR_NULO_REQUERIDO",
            "descripcion": f"Registros sin id_equipo: {sin_equipo}",
            "columna": "id_equipo",
            "severidad": "ALTA",
            "accion_recomendada": "Investigar causa y asignar id_equipo correcto"
        })
    
    # Hallazgo 2: Registros sin fecha_fin que no están EN_PROCESO
    mask = (self.df["fecha_fin"].isna()) & (self.df["estado"] != "EN_PROCESO")
    sin_fecha_fin = mask.sum()
    if sin_fecha_fin > 0:
        hallazgos.append({
            "tipo": "INCONSISTENCIA_ESTADO",
            "descripcion": f"Registros sin fecha_fin que no están EN_PROCESO: {sin_fecha_fin}",
            "columna": "fecha_fin",
            "severidad": "ALTA",
            "accion_recomendada": "Verificar si el trabajo está realmente en proceso"
        })
    
    # Hallazgo 3: horas_trabajo fuera del rango [0-24]
    horas_invalidas = ((self.df["horas_trabajo"] < 0) | 
                       (self.df["horas_trabajo"] > 24)).sum()
    if horas_invalidas > 0:
        hallazgos.append({
            "tipo": "VALOR_FUERA_RANGO",
            "descripcion": f"Registros con horas_trabajo fuera de [0-24]: {horas_invalidas}",
            "columna": "horas_trabajo",
            "severidad": "MEDIA",
            "accion_recomendada": "Revisar y corregir valores de horas"
        })
    
    # Hallazgo 4: tipo_mantenimiento no válido
    tipos_validos = {"PREVENTIVO", "CORRECTIVO", "PREDICTIVO"}
    tipos_invalidos = ~self.df["tipo_mantenimiento"].isin(tipos_validos)
    num_tipos_invalidos = tipos_invalidos.sum()
    if num_tipos_invalidos > 0:
        hallazgos.append({
            "tipo": "VALOR_INVALIDO",
            "descripcion": f"Registros con tipo_mantenimiento no válido: {num_tipos_invalidos}",
            "columna": "tipo_mantenimiento",
            "severidad": "MEDIA",
            "accion_recomendada": "Normalizar tipos de mantenimiento"
        })
    
    return hallazgos

# Añadir método a la clase
DataProfiler.identificar_hallazgos = identificar_hallazgos

### 2.4 Generación de Informes

Métodos para generar el informe completo y exportarlo en múltiples formatos.

In [ ]:
def generar_informe(self) -> Dict[str, Any]:
    """Genera el informe completo de profiling."""
    logger.info("Generando informe de profiling...")
    
    informe = {
        "metadata": self.generar_metadata(),
        "resumen_ejecutivo": self.calcular_resumen_ejecutivo(),
        "analisis_columnas": [],
        "hallazgos": self.identificar_hallazgos()
    }
    
    # Analizar cada columna
    for col in self.df.columns:
        informe["analisis_columnas"].append(self.analizar_columna(col))
    
    return informe

def guardar_json(self, filepath: str):
    """Guarda el informe en formato JSON."""
    informe = self.generar_informe()
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(informe, f, indent=2, ensure_ascii=False)
    logger.info(f"Informe JSON guardado: {filepath}")

def guardar_markdown(self, filepath: str):
    """Guarda el informe en formato Markdown."""
    informe = self.generar_informe()
    md = f"# Informe de Profiling: {self.source_filename}\n\n"
    md += f"**Fecha de generación:** {informe['metadata']['fecha_generacion']}\n\n"
    
    # Resumen ejecutivo
    md += "## Resumen Ejecutivo\n\n"
    md += f"- **Total registros:** {informe['resumen_ejecutivo']['total_registros']}\n"
    md += f"- **Total columnas:** {informe['resumen_ejecutivo']['total_columnas']}\n"
    md += f"- **Completitud global:** {informe['resumen_ejecutivo']['completitud_global_pct']}%\n"
    md += f"- **Score de calidad:** {informe['resumen_ejecutivo']['score_calidad']}/100\n\n"
    
    # Análisis de columnas
    md += "## Análisis por Columna\n\n"
    for col in informe['analisis_columnas']:
        md += f"### {col['nombre']}\n\n"
        md += f"- **Tipo:** {col['tipo_dato']}\n"
        md += f"- **Nulos:** {col['nulos_count']} ({col['nulos_pct']}%)\n"
        md += f"- **Únicos:** {col['unicos']}\n"
        md += f"- **Completitud:** {col['completitud_pct']}%\n"
        if 'min' in col:
            md += f"- **Rango:** {col['min']} - {col['max']}\n"
            md += f"- **Media:** {col['media']:.2f}\n"
            md += f"- **Mediana:** {col['mediana']:.2f}\n"
        if 'top_5_valores' in col:
            md += "- **Top 5 valores:**\n"
            for v in col['top_5_valores']:
                md += f"  - {v['valor']}: {v['conteo']}\n"
        md += "\n"
    
    # Hallazgos
    md += "## Hallazgos de Calidad\n\n"
    if informe['hallazgos']:
        for h in informe['hallazgos']:
            md += f"### {h['tipo']} - {h['columna']}\n\n"
            md += f"- **Descripción:** {h['descripcion']}\n"
            md += f"- **Severidad:** {h['severidad']}\n"
            md += f"- **Acción:** {h['accion_recomendada']}\n\n"
    else:
        md += "No se encontraron hallazgos de calidad.\n\n"
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(md)
    logger.info(f"Informe Markdown guardado: {filepath}")

# Añadir métodos a la clase
DataProfiler.generar_informe = generar_informe
DataProfiler.guardar_json = guardar_json
DataProfiler.guardar_markdown = guardar_markdown

---

## 3. Configuración de Conexión a S3

Configuramos la conexión a S3 y definimos las rutas de origen (landing) y destino (profiling reports).

In [ ]:
# Configuración de S3
BUCKET_NAME = "<TU_BUCKET_NAME>"
LANDING_PREFIX = "landing/obra-lineal"
PROFILE_REPORTS_PREFIX = "profile-reports/obra-lineal"

# Cliente S3
s3_client = boto3.client("s3")

# Directorio local temporal para informes
LOCAL_OUTPUT_DIR = "./profile_reports"
Path(LOCAL_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

logger.info(f"✅ Configuración S3:")
logger.info(f"   Bucket: {BUCKET_NAME}")
logger.info(f"   Landing: {LANDING_PREFIX}")
logger.info(f"   Reports: {PROFILE_REPORTS_PREFIX}")

## 4. Listar Archivos en Landing Zone

Exploramos los archivos disponibles en la zona landing de S3.

In [ ]:
# Listar archivos CSV en landing
logger.info(f"Listando archivos en s3://{BUCKET_NAME}/{LANDING_PREFIX}/...")

response = s3_client.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=LANDING_PREFIX
)

# Filtrar solo archivos CSV (excluir metadata)
csv_files = [
    obj['Key'] for obj in response.get('Contents', [])
    if obj['Key'].endswith('.csv')
]

print(f"\n📁 Archivos CSV encontrados en landing ({len(csv_files)}):")
for i, key in enumerate(csv_files[:5], 1):  # Mostrar máximo 5
    size_mb = response['Contents'][i-1]['Size'] / (1024 * 1024)
    modified = response['Contents'][i-1]['LastModified']
    print(f"  {i}. {key.split('/')[-1]} ({size_mb:.2f} MB) - {modified}")

if len(csv_files) == 0:
    print("⚠️  No se encontraron archivos CSV. Ejecuta primero el notebook de ingesta.")
else:
    # Seleccionar el archivo más reciente
    latest_file = sorted(csv_files, reverse=True)[0]
    logger.info(f"Archivo seleccionado: {latest_file}")

## 5. Cargar Datos desde S3

Usamos AWS Data Wrangler para leer el archivo CSV directamente desde S3.

In [ ]:
# Cargar datos desde S3
if len(csv_files) > 0:
    s3_path = f"s3://{BUCKET_NAME}/{latest_file}"
    logger.info(f"📥 Cargando datos desde {s3_path}...")
    
    # Leer CSV desde S3 usando awswrangler
    df = wr.s3.read_csv(path=s3_path)
    
    logger.info(f"✅ Datos cargados: {len(df)} registros, {len(df.columns)} columnas")
    
    # Extraer nombre del archivo para el informe
    source_filename = latest_file.split('/')[-1]
    
    # Visualizar muestra
    print("\n📊 Primeras filas del dataset:")
    display(df.head())
    
    print("\n📋 Información del dataset:")
    df.info()
else:
    raise FileNotFoundError("No hay archivos en landing. Ejecuta primero el notebook de ingesta.")

## 6. Generar Informe de Profiling

Ejecutamos el pipeline completo de profiling sobre los datos cargados desde S3.

In [ ]:
# Generar profiling
logger.info("Iniciando profiling de datos...")
profiler = DataProfiler(df, source_filename)

# Generar nombres de archivos para informes
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
json_filename = f"profiling_{timestamp}.json"
md_filename = f"profiling_{timestamp}.md"

# Rutas locales
json_path = f"{LOCAL_OUTPUT_DIR}/{json_filename}"
md_path = f"{LOCAL_OUTPUT_DIR}/{md_filename}"

# Guardar informes localmente
profiler.guardar_json(json_path)
profiler.guardar_markdown(md_path)

logger.info("✅ Informes generados localmente")

In [ ]:
# Subir informes a S3
logger.info("Subiendo informes a S3...")

# Estructura de fechas para organización
ahora = datetime.now()
fecha_path = f"año={ahora.year}/mes={ahora.month:02d}/dia={ahora.day:02d}"

# Claves S3
s3_json_key = f"{PROFILE_REPORTS_PREFIX}/{fecha_path}/{json_filename}"
s3_md_key = f"{PROFILE_REPORTS_PREFIX}/{fecha_path}/{md_filename}"

# Upload JSON
s3_client.upload_file(
    json_path,
    BUCKET_NAME,
    s3_json_key,
    ExtraArgs={'ContentType': 'application/json'}
)
logger.info(f"✅ JSON subido: s3://{BUCKET_NAME}/{s3_json_key}")

# Upload Markdown
s3_client.upload_file(
    md_path,
    BUCKET_NAME,
    s3_md_key,
    ExtraArgs={'ContentType': 'text/markdown'}
)
logger.info(f"✅ Markdown subido: s3://{BUCKET_NAME}/{s3_md_key}")

print(f"\n📤 Informes guardados en S3:")
print(f"   • JSON: s3://{BUCKET_NAME}/{s3_json_key}")
print(f"   • Markdown: s3://{BUCKET_NAME}/{s3_md_key}")

## 7. Subir Informes a S3

Guardamos los informes de profiling en S3 junto a los datos analizados.

## 8. Visualización del Resumen Ejecutivo

In [ ]:
# Obtener informe completo
informe = profiler.generar_informe()

# Mostrar resumen ejecutivo
print("\n" + "="*60)
print("📈 RESUMEN EJECUTIVO - PROFILING DE DATOS")
print("="*60)
print(f"\n📁 Archivo: {informe['metadata']['nombre_archivo']}")
print(f"📅 Fecha: {informe['metadata']['fecha_generacion']}")
print(f"\n📊 Registros: {informe['resumen_ejecutivo']['total_registros']:,}")
print(f"🗂️  Columnas: {informe['resumen_ejecutivo']['total_columnas']}")
print(f"✅ Completitud global: {informe['resumen_ejecutivo']['completitud_global_pct']:.2f}%")
print(f"⭐ Score de calidad: {informe['resumen_ejecutivo']['score_calidad']}/100")
print("\n" + "="*60)

## 9. Visualización de Hallazgos de Calidad

In [ ]:
# Mostrar hallazgos
print("\n" + "="*60)
print("🔍 HALLAZGOS DE CALIDAD")
print("="*60)

if informe['hallazgos']:
    for i, hallazgo in enumerate(informe['hallazgos'], 1):
        print(f"\n[{i}] {hallazgo['tipo']} - Columna: {hallazgo['columna']}")
        print(f"    📝 {hallazgo['descripcion']}")
        print(f"    ⚠️  Severidad: {hallazgo['severidad']}")
        print(f"    💡 Acción: {hallazgo['accion_recomendada']}")
else:
    print("\n✅ No se encontraron hallazgos de calidad.")

print("\n" + "="*60)

## 10. Análisis Detallado por Columna (Top 3)

In [ ]:
# Mostrar análisis de las 3 primeras columnas
print("\n" + "="*60)
print("📊 ANÁLISIS DETALLADO POR COLUMNA (Top 3)")
print("="*60)

for col in informe['analisis_columnas'][:3]:
    print(f"\n🔹 {col['nombre']}")
    print(f"   Tipo: {col['tipo_dato']}")
    print(f"   Nulos: {col['nulos_count']} ({col['nulos_pct']}%)")
    print(f"   Únicos: {col['unicos']}")
    print(f"   Completitud: {col['completitud_pct']}%")
    
    if 'min' in col:
        print(f"   Rango: [{col['min']}, {col['max']}]")
        print(f"   Media: {col['media']:.2f}")
    
    if 'top_5_valores' in col:
        print("   Top valores:")
        for v in col['top_5_valores'][:3]:
            print(f"     - {v['valor']}: {v['conteo']}")

print("\n" + "="*60)
print(f"\n💾 Informes guardados:")
print(f"   📂 Local:")
print(f"      • JSON: {json_path}")
print(f"      • Markdown: {md_path}")
print(f"   ☁️  S3:")
print(f"      • JSON: s3://{BUCKET_NAME}/{s3_json_key}")
print(f"      • Markdown: s3://{BUCKET_NAME}/{s3_md_key}")